# RT-100 streaming (single booster, 500 causal features)

Structural Break Challenge -- Real-Time Edition.

**Architecture.** One shared incremental feature engine (`sbr.stream.StreamEngine`)
updates 500 causal features per observation in O(1) amortised time, feeding
1 LightGBM booster(s).

**Causality.** Every feature module is bitwise-identical to a batch implementation
that passes prefix-invariance at `atol=0`, and the full 500-column vector is
bitwise-identical between the streaming and batch paths on real and adversarial
series (`tests/test_stream_engine_parity.py`). Streaming predictions match batch
predictions to 0.0 over 12,727 points end to end.

**No cross-series information.** Scores depend only on this series' history and
its online points up to the current index. `INFER_PARALLELISM` is safe precisely
because there is no shared state; the local harness asserts that shuffling the
series order leaves every score unchanged.

**`n_online` is never observed.** Asserted by `tests/test_production_contract.py::test_n_online_is_never_a_feature`.

Built from git `b5ea9d1d9c`, feature manifest `1646c3b9e09d8a7f`.


In [ ]:
# ---------------------------------------------------------------------------
# SELF-EXTRACTING PAYLOAD -- generated by research/scripts/build_submission.py
# Do not edit by hand.  Rebuild instead:  python3 research/scripts/build_submission.py
#
#   source zip sha256 : 32e7f9638375144bd976be40a800e38503a2d73fbd9eec9e7dc4784cc53ddf2f
#   model  zip sha256 : d48dea480c7a25c4f08a07bc0ea318d8ffdc80ce79f0add4fb298430138a4629
#   feature manifest  : 1646c3b9e09d8a7fb3c564483a1d1d999680caeefe848b092f907a6cac80cced
#   code git sha      : b5ea9d1d9cd87f06f574c5a78e4c850f41ef852f
#   built at          : 2026-08-19T07:23:21Z
# ---------------------------------------------------------------------------
import base64, hashlib, io, os, sys, zipfile

_SRC_B64 = "..."
_MDL_B64 = "..."

_WORK = os.path.abspath("./_sbr_payload")
os.makedirs(_WORK, exist_ok=True)


def _unpack(b64, expect_sha, name):
    raw = base64.b64decode(b64)
    got = hashlib.sha256(raw).hexdigest()
    if got != expect_sha:
        raise RuntimeError(f"{name} payload corrupted: {got} != {expect_sha}")
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        z.extractall(_WORK)
    return got


_unpack(_SRC_B64, "32e7f9638375144bd976be40a800e38503a2d73fbd9eec9e7dc4784cc53ddf2f", "source")
_unpack(_MDL_B64, "d48dea480c7a25c4f08a07bc0ea318d8ffdc80ce79f0add4fb298430138a4629", "model")
if _WORK not in sys.path:
    sys.path.insert(0, _WORK)

MODEL_DIRECTORY = os.path.join(_WORK, "model")
print("payload ok; source and model verified by sha256")


In [ ]:
import os
from typing import Iterable, Iterator, List, Optional, Tuple

import numpy as np

from sbr.production.model import ProductionModel

# Series are independent by construction -- there is no cross-series state
# anywhere in this submission, which is exactly why parallel inference is safe.
# @crunch/keep:on
INFER_PARALLELISM = 4

_MODEL = None


def train(datasets, model_directory_path: str):
    """Install the pre-trained, pre-validated model.

    The model is a frozen function of the training data and was fitted offline
    on the 8,000 development series with the full 500-column causal feature bank
    (see research/REPRODUCIBILITY_MANIFEST.json).  Refitting it inside the
    runner would add an hour of feature construction and a large amount of
    avoidable risk for no change in the resulting function.
    """
    import shutil
    os.makedirs(model_directory_path, exist_ok=True)
    for f in os.listdir(MODEL_DIRECTORY):
        shutil.copy2(os.path.join(MODEL_DIRECTORY, f), os.path.join(model_directory_path, f))
    ProductionModel.load(model_directory_path)      # raises on any mismatch


def infer(datasets: Iterable[Tuple[List[float], Iterable[float]]],
          model_directory_path: str):
    """One score per online observation, in order, single pass, causal.

    At online index t this uses only: this series' historical segment and this
    series' online points 0..t.  No other series, no future point, and never
    the online length -- which is unknowable at t and was the leak that
    invalidated the pre-2026-06-08 predictions.
    """
    model = ProductionModel.load(model_directory_path)

    yield                                      # readiness handshake

    for x_historical, x_online in datasets:
        model.start_series(np.asarray(x_historical, dtype=np.float64))
        for point in x_online:
            yield model.step(float(point))
